# 02 — Prédire l'insatisfaction à l'atterrissage

Responsable : **Jeanne** — branche `modeles`

Données : **train uniquement**, comparaison des modèles en validation croisée.

`test.csv` n'est **jamais** lu ici : voir le notebook 03.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

from src.data import load_train, TARGET, RANDOM_STATE
from src.features import LANDING_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.economics import gain_net, BREAK_EVEN_PRECISION

df = load_train()
X = df[LANDING_FEATURES]
y = df[TARGET]
X.shape, y.mean()

((103904, 7), np.float64(0.5666673082845703))

## 1. Variables connues à l'atterrissage
Justification de chaque variable retenue ou exclue (voir `src/features.py`).

## 2. Méthode de comparaison
Validation croisée stratifiée à 5 plis, `random_state=42`, pipeline (encodage + imputation du retard d'arrivée).

## 3. Règle simple sans modèle (baseline)
Hasard (10 % des passagers) et règle métier.

In [2]:
import numpy as np

rng = np.random.default_rng(RANDOM_STATE)   # hasard reproductible : même tirage à chaque exécution
scores_hasard = rng.random(len(y))          # un score aléatoire par passager

gain_net(y, scores_hasard)

{'passagers_abordes': 10390,
 'insatisfaits_abordes': 5876,
 'precision': 0.5655437921077959,
 'rappel': 0.0997978905891744,
 'gain_net': 172000.0,
 'gain_net_pour_1000': 1655.3741915614412}

## 4. Modèle 1 — régression logistique

In [4]:
# Retard d'arrivée manquant (310 passagers) : on le remplace par le retard au départ,
# les deux sont quasi identiques (corrélation 0,97)
retard = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])

# Règle 1 : aborder les plus gros retards à l'arrivée
regle_retard = gain_net(y, retard)

# Règle 2 : voyage perso d'abord, puis les plus gros retards
perso = (X["Type of Travel"] == "Personal Travel").astype(int)
regle_perso = gain_net(y, perso * 10_000 + retard)

import pandas as pd
baselines = pd.DataFrame(
    [gain_net(y, scores_hasard), regle_retard, regle_perso],
    index=["Hasard", "Règle : plus gros retards", "Règle : voyage perso puis retard"],
)
baselines[["precision", "rappel", "gain_net_pour_1000"]].round(3)

,precision,rappel,gain_net_pour_1000
Hasard,0.566,0.100,1655.374
Règle : plus gros retards,0.642,0.113,2422.428
Règle : voyage perso puis retard,0.956,0.169,5564.752


## 5. Modèle 2 — forêt aléatoire / gradient boosting

## 6. Comparaison et choix du modèle
Précision dans le top 10 %, gain net pour 1 000 passagers, coût des erreurs, lisibilité.